# Project 2 — Preflight tests (small data)

Run this notebook **before** the full 8-config grid in `proj2-scratch.ipynb`.

Goals:
1. Confirm `train.json` / `validation.json` rows have `schema_links`
2. Reproduce the **double `map(formatting_func)`** issue RapidFire can trigger
3. Verify the **idempotent** formatter in `schema_formatting.py` fixes it
4. Optional: one tiny `run_fit` on 8 train / 4 val examples

## Setup

In [ ]:
import json
import pickle
from pathlib import Path

import cloudpickle
from datasets import Dataset

from paths import TRAIN_JSON, VAL_INPUT_JSON, VAL_JSON, require_release_data, setup

setup(chdir_try=True)
require_release_data()

from schema_formatting import (
    format_row,
    make_formatting_func,
    sort_schema_links,
)
from schema_metrics import schema_link_metrics

TRAIN_PATH = TRAIN_JSON
VAL_PATH = VAL_JSON
VAL_INPUT_PATH = VAL_INPUT_JSON

with open(TRAIN_PATH) as f:
    raw_train = json.load(f)
with open(VAL_PATH) as f:
    raw_val = json.load(f)
with open(VAL_INPUT_PATH) as f:
    raw_val_input = json.load(f)

# Small subsets for fast iteration (increase for 8-config smoke tests)
# Keep this much smaller than full data so all configs finish quickly.
N_TRAIN, N_VAL = 64, 32
tiny_train = raw_train[:N_TRAIN]
tiny_val = raw_val[:N_VAL]

train_ds = Dataset.from_list(tiny_train)
val_ds = Dataset.from_list(tiny_val)

fmt = make_formatting_func("minimal")

print(f"Tiny train: {len(tiny_train)} | Tiny val: {len(tiny_val)}")
print("Train keys:", list(tiny_train[0].keys()))

Tiny train: 64 | Tiny val: 32
Train keys: ['question_id', 'db_id', 'question', 'gold_sql', 'schema_links']


## Test 1 — Raw JSON has `schema_links`

In [ ]:
def check_schema_links(name, rows):
    bad = [ex for ex in rows if ex.get("schema_links") is None]
    missing = [ex for ex in rows if "schema_links" not in ex]
    print(f"{name}: total={len(rows)} None={len(bad)} missing_key={len(missing)}")
    if bad:
        print("  example None:", bad[0].get("question_id"))
    return len(bad) == 0 and len(missing) == 0

assert check_schema_links("train.json slice", tiny_train)
assert check_schema_links("validation.json slice", tiny_val)
print("validation_input (no labels — expected fail for training):")
check_schema_links("validation_input.json slice", raw_val_input[:N_VAL])

## Test 2 — Format once (should always work)

In [3]:
sample = fmt(tiny_train[0])
print("roles:", [m["role"] for m in sample["prompt"]], sample["completion"][0]["role"])
print("target:", sample["completion"][0]["content"])
print("user preview:", sample["prompt"][1]["content"], "...")

roles: ['system', 'user'] assistant
target: {"INJURY": ["AIS", "REGION"]}
user preview: Database schema:
AIRBAG(CASEID, PSU, CASENO, CASENUMBER, CATEGORY, VEHNO, BAGNO, BAGLOCATION, BAGSTATUS, BAGTYPE, BAGDEPLOY, BAGMALFUNCTION, BAGFLAPSOPEN, BAGFLAPSDAM, BAGDAMAGE, BAGDAMSOURCE, PREVCRASH, PRIORMAINT, DEPLOYEVENT, BAGDVTOTAL, BAGDVLONG, BAGDVLAT, BAGDVENERGY, BAGDVSPEED, BAGDVBES, BAGDVEST, BAGDVRANK, CDCFORDEPLOY, CASEWGT, PSUSTRAT, VERSION)
CHILDSEAT(CASEID, PSU, CASENO, CASENUMBER, CATEGORY, VEHNO, OCCNO, CHILDSEATNO, SEATLOC, CHILDMANUF, CHILDMAKE, CHILDMODEL, CHILDMODELNO, CHILDSEATYPE, CHILDDATEMAN, HOWUSED, DATASOURCE, ORIENTATION, HARNESSDESIGN, RETAINERCLIPDESIGN, TETHERDESIGN, LATCHDESIGN, HARNESSUSE, RETAINERCLIPUSE, TETHERUSE, LATCHUSE, BELTROUT, LOCKCLIPUSE, PLACEMENT, CHILDPOSITION, BELTRETYPE, LATCHPLATE, LATCHANCHOR, LATCHTETHER, CASEWGT, PSUSTRAT, VERSION)
CRASH(CASEID, CRASHYEAR, PSU, CASENO, CASENUMBER, CATEGORY, CRASHMONTH, DAYOFWEEK, CRASHTIME, EVENTS, VEHICLES, C

## Test 3 — Double format (reproduces RapidFire issue)

RapidFire may call `dataset.map(formatting_func)` more than once.  
A **non-idempotent** formatter tries to read `schema_links` again and crashes.

In [4]:
def legacy_formatting_func(row):
    """Old nested style — NOT idempotent (for demonstration)."""
    from schema_formatting import load_schema, serialize_schema_minimal, SYSTEM_PROMPT
    schema_s = serialize_schema_minimal(load_schema(row["db_id"]))
    user_msg = f"Database schema:\n{schema_s}\n\nQuestion: {row['question']}"
    target = json.dumps(sort_schema_links(row["schema_links"]), ensure_ascii=False)
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ],
        "completion": [{"role": "assistant", "content": target}],
    }

mapped_once = train_ds.map(legacy_formatting_func)
print("After 1st map, columns:", mapped_once.column_names)
print("schema_links still present?", "schema_links" in mapped_once.column_names)

# HuggingFace keeps original columns, so a 2nd .map() often still works.
# RapidFire / TRL may pass rows with ONLY prompt+completion — simulate that:
stripped = {
    "prompt": mapped_once[0]["prompt"],
    "completion": mapped_once[0]["completion"],
}
try:
    legacy_formatting_func(stripped)
    print("legacy on stripped row: OK (unexpected)")
except Exception as e:
    print("legacy on stripped row FAILED (this is the real bug):", type(e).__name__, e)

try:
    fmt(stripped)
    print("idempotent fmt on stripped row: OK")
except Exception as e:
    print("idempotent fmt on stripped row FAILED:", type(e).__name__, e)

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

After 1st map, columns: ['question_id', 'db_id', 'question', 'gold_sql', 'schema_links', 'prompt', 'completion']
schema_links still present? True
legacy on stripped row FAILED (this is the real bug): KeyError 'db_id'
idempotent fmt on stripped row: OK


In [5]:
mapped_once = train_ds.map(fmt)
try:
    mapped_twice = mapped_once.map(fmt)
    row = mapped_twice[0]
    print("idempotent fmt: 2nd map OK")
    print("  still has prompt/completion:", "prompt" in row, "completion" in row)
except Exception as e:
    print("idempotent fmt: 2nd map FAILED:", type(e).__name__, e)

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

idempotent fmt: 2nd map OK
  still has prompt/completion: True True


## Test 4 — Pickle / cloudpickle (worker simulation)

In [6]:
row = tiny_train[0]

# cloudpickle — closer to what multiprocessing uses
fmt_remote = cloudpickle.loads(cloudpickle.dumps(fmt))
out = fmt_remote(row)
print("cloudpickle round-trip OK, target:", out["completion"][0]["content"])

# stdlib pickle should also work with partial(format_row, ...)
fmt_p = pickle.loads(pickle.dumps(fmt))
print("pickle round-trip OK:", fmt_p(row)["completion"][0]["content"])

cloudpickle round-trip OK, target: {"INJURY": ["AIS", "REGION"]}
pickle round-trip OK: {"INJURY": ["AIS", "REGION"]}


## Test 5 — `validation_input` must not be used for training eval

In [7]:
bad_ds = Dataset.from_list(raw_val_input[:N_VAL])
try:
    bad_ds.map(fmt)[0]
    print("ERROR: should have failed")
except Exception as e:
    print("Expected failure on validation_input:", type(e).__name__)
    print(" ", e)

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Expected failure on validation_input: ValueError
  schema_links missing on row. keys=['question_id', 'db_id', 'question'] question_id=1. Use validation.json (not validation_input.json) for training eval.


## Test 6 — Metrics smoke test

In [8]:
gold = '{"INJURY": ["AIS", "REGION"]}'
print("perfect:", schema_link_metrics([gold], [gold]))
print("partial:", schema_link_metrics(['{"INJURY": ["AIS"]}'], [gold]))

perfect: {'table_P': 1.0, 'table_R': 1.0, 'table_F1': 1.0, 'col_P': 1.0, 'col_R': 1.0, 'col_F1': 1.0, 'leaderboard_score': 1.0}
partial: {'table_P': 1.0, 'table_R': 1.0, 'table_F1': 1.0, 'col_P': 1.0, 'col_R': 0.5, 'col_F1': 0.6667, 'leaderboard_score': 0.8611}


## Test 7 — `SFTConfig` builds (RapidFire-compatible args)

In [9]:
from rapidfireai.automl import RFSFTConfig

cfg = RFSFTConfig(
    learning_rate=1e-4,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    max_steps=16,
    logging_steps=4,
    eval_strategy="steps",
    eval_steps=8,
    load_best_model_at_end=False,
    max_length=1024,
    bf16=True,
    report_to="none",
)
print("SFTConfig OK:", cfg.max_length, cfg.eval_strategy, cfg.load_best_model_at_end)

SFTConfig OK: 1024 IntervalStrategy.STEPS False


## Test 8 — Optional tiny `run_fit` (GPU)

Set `RUN_TINY_FIT = True` to run one short training job (~5–15 min).  
Uses a **new** experiment name so it does not clash with the main sweep.

In [11]:
RUN_TINY_FIT = True  # flip to True when ready

if RUN_TINY_FIT:
    from rapidfireai import Experiment
    from rapidfireai.automl import List, RFGridSearch, RFModelConfig, RFLoraConfig, RFSFTConfig
    from schema_metrics import compute_metrics_for_rf

    def create_model(model_config: dict):
        from transformers import AutoModelForCausalLM, AutoTokenizer
        model_name = model_config["model_name"]
        model_kwargs = model_config["model_kwargs"]
        model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
        tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
            model.config.pad_token_id = tokenizer.eos_token_id
        return model, tokenizer

    experiment = Experiment(
        experiment_name="proj2-preflight-tiny",
        mode="fit",
        experiment_path="./logs",
    )

    peft_narrow = RFLoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, target_modules=["q_proj", "v_proj"], bias="none")

    tiny_config = List([
        RFModelConfig(
            model_name="Qwen/Qwen2.5-0.5B-Instruct",
            peft_config=peft_narrow,
            training_args=RFSFTConfig(
                learning_rate=1e-4,
                per_device_train_batch_size=2,
                per_device_eval_batch_size=2,
                max_steps=20,
                logging_steps=5,
                eval_strategy="steps",
                eval_steps=10,
                load_best_model_at_end=False,
                max_length=1024,
                bf16=True,
                report_to="none",
            ),
            model_type="causal_lm",
            model_kwargs={"device_map": "auto", "torch_dtype": "auto", "use_cache": False},
            formatting_func=fmt,
            compute_metrics=compute_metrics_for_rf,
            generation_config={"max_new_tokens": 128, "do_sample": False},
        )
    ])

    group = RFGridSearch(configs=tiny_config, trainer_type="SFT")
    experiment.run_fit(group, create_model, train_ds, val_ds, num_chunks=2, seed=42)
    print("Tiny run_fit finished — check ./logs/proj2-preflight-tiny/")
else:
    print("Skipped tiny run_fit (set RUN_TINY_FIT = True to enable)")

The previously running experiment proj2-schema-linking-v1 was forcibly ended. Created a new experiment 'proj2-preflight-tiny' with Experiment ID: 12 and Metric Experiment ID: 33 at /home/emc001/schema-linker-sft-pipeline/logs/proj2-preflight-tiny
INFO 05-28 12:38:33 [__init__.py:216] Automatically detected platform cuda.
Started 1 worker processes successfully
Created workers
INFO 05-28 12:38:48 [__init__.py:216] Automatically detected platform cuda.
Tiny run_fit finished — check ./logs/proj2-preflight-tiny/


## Checklist before full grid

- [ ] Test 1 passes on tiny slices
- [ ] Test 3: legacy fails on 2nd map, idempotent `fmt` passes
- [ ] Test 4: cloudpickle + pickle round-trip OK
- [ ] Test 5: `validation_input` fails as expected
- [ ] Test 7: `SFTConfig` constructs without error
- [ ] (Optional) Test 8 tiny `run_fit` completes with no `Run N has failed`

Then in `proj2-scratch.ipynb` use:
```python
from schema_formatting import make_formatting_func
from schema_metrics import compute_metrics_for_rf
fmt_func = make_formatting_func("minimal")
```